In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
import os

api_key = os.getenv("GOOGLE_API_KEY")

if os.environ['GOOGLE_API_KEY']:
    print("Google api key is set")
else:
    raise ValueError("Gemini API key is not set")


llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", google_api_key=api_key, temperature=0.7)

c:\FAHEEM\My_Programs\RAG_Beginners\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Google api key is set


In [ ]:
from langchain_core.tools import tool
from gmail_service import get_gmail_service
#get_email_thread(thread_id)
@tool
def get_new_emails(max_results = 10):
    """
    Use this tool to access email and fetch new emails
    """
    service = get_gmail_service()

    results = service.users().messages().list(
        userId="me",
        maxResults=max_results
    ).execute()

    messages = results.get("messages", [])

    emails = []

    for message in messages:
        email = service.users().messages().get(
            userId="me",
            id=message["id"],
            format="full"
        ).execute()

        headers = email["payload"].get("headers", [])

        subject = ""
        sender = ""
        date = ""

        for header in headers:
            if header["name"] == "Subject":
                subject = header["value"]

            elif header["name"] == "From":
                sender = header["value"]

            elif header["name"] == "Date":
                date = header["value"]

        emails.append({
            "id": message["id"],
            "sender": sender,
            "subject": subject,
            "date": date
        })

    return emails

@tool
def search_internal_docs(query):
    """
    Use this tool to access internal documents required for email related issue or reply

    Args:
        query: To elaborate what type of document needed to be fetched
    """

@tool
def search_memory(query):
    """
    Use this tool to load previous conversation to handle to validate user requests

    Args:
        query: To elaborate what type of data to be loaded from memory for further process  
    """

@tool
def send_email(to, subject, body):
    """
    Use this tool to reply to customer or clients by generating a proper professional email

    Args:
        to: Holds email of user to whom email is to be send
        subject: Subject of the email
        body: content of the email
    """

In [ ]:
tools = [get_new_emails, search_internal_docs, search_memory, send_email]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
from typing import TypedDict, Annotated, List
from langgraph.graph.message import BaseMessage, add_messages


class graph_schema(TypedDict):
    query: Annotated[List[BaseMessage], add_messages]
    answer: str

In [ ]:
from langchain_core.messages import ToolMessage, AIMessage

def llm_node(state: graph_schema) -> graph_schema:
    query = state['query']
    response = llm_with_tools.invoke(query)

    print(response)
    return {
        "query": response
    }

def tool_node(state: graph_schema) -> graph_schema:

    query = state['query'][-1]

    tool_name = {tool.name: tool for tool in tools}

    tool_result = []

    for tool_call in query.tool_calls:
        tool = tool_name[tool_call["name"]]

        observation = tool.invoke(tool_call['args'])

        tool_result.append(ToolMessage(content=str(observation), tool_call_id=tool_call['id']))

    return {
        'answer': tool_result
    }

def tool_decision(state: graph_schema) -> str:
    if state['query'][-1].tool_calls:
        return "tool_node"
    else:
        return "end"


In [1]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(graph_schema)

graph.add_node("llm_node", llm_node)
graph.add_node("tool_node", tool_node)

graph.add_edge(START, "llm_node")
graph.add_conditional_edges("llm_node", tool_decision, {"tool_node": "tool_node", "end": END})
graph.add_edge("tool_node", "llm_node")
graph.add_edge("llm_node", END)

email_agent = graph.compile()
email_agent

NameError: name 'graph_schema' is not defined

In [ ]:
conversation = []

while True:
    query = input("Enter a query to proceed (stop to end task...)")

    if query == "stop":
        break

    conversation.append(AIMessage(content=query))

    response = email_agent.invoke({
        "query": conversation,
        "answer": ""
    })

    print(response)
